# Lecture 17: Constrained Optimization — Barrier Method

---

```{note}
Lecture 16 introduced the Penalty Method: append a penalty for constraint violations to the objective and increase its weight until the optimizer stops violating. The key limitation is that Penalty Method iterates are infeasible throughout — they approach the constraint boundary from outside. This lecture introduces the Barrier Method, which takes the opposite approach: start strictly inside the feasible region and add a term that becomes infinitely large as any constraint boundary is approached, preventing the optimizer from ever leaving the feasible set.
```

**Learning Objectives**

By the end of this notebook, you will be able to:
1. State the Karush–Kuhn–Tucker (KKT) conditions and identify what each condition requires of a constrained optimum.
2. Formulate the logarithmic barrier function for an inequality-constrained NLP and derive the barrier stationarity conditions.
3. Trace the Barrier Method iteration by hand and explain why the constrained optimum is recovered only in the limit as the barrier parameter $\mu \to 0$.

**Prerequisites**: NLP Principles (Lecture 12); Gradient Descent, Newton's Method, BFGS (Lectures 13–15); Penalty Method (Lecture 16); calculus (partial derivatives, logarithmic differentiation); basic linear algebra (solving 2×2 systems).

**Estimated time**: 90 minutes (including in-class exercises)

---

## Optimality Conditions — KKT

Consider the standard-form NLP:

$$\min_{x} \; f(x) \quad \text{subject to} \quad g_i(x) \leq 0, \quad i = 1, \ldots, m$$

where $f, g_i : \mathbb{R}^n \to \mathbb{R}$ are continuously differentiable. The **Lagrangian** is:

$$\mathcal{L}(x, \lambda) = f(x) + \sum_{i=1}^{m} \lambda_i \, g_i(x)$$

If $x^*$ is a local minimum and a constraint qualification holds (e.g., LICQ), then there exist multipliers $\lambda^* \in \mathbb{R}^m$ such that the following **Karush–Kuhn–Tucker (KKT) conditions** are satisfied:

| Condition | Statement | Interpretation |
|-----------|-----------|----------------|
| **Stationarity** | $\nabla f(x^*) + \sum_{i=1}^{m} \lambda_i^* \nabla g_i(x^*) = \mathbf{0}$ | Gradient of objective balanced by gradients of active constraints |
| **Primal feasibility** | $g_i(x^*) \leq 0 \quad \forall\, i$ | Solution lies within the feasible region |
| **Dual feasibility** | $\lambda_i^* \geq 0 \quad \forall\, i$ | Multipliers are non-negative for inequality constraints |
| **Complementary slackness** | $\lambda_i^* \, g_i(x^*) = 0 \quad \forall\, i$ | A constraint is either active ($g_i = 0$) or its multiplier is zero ($\lambda_i = 0$) |

For a **convex** NLP (convex $f$, convex $g_i$), the KKT conditions are both necessary and sufficient for global optimality. All methods in Lectures 16–18 seek a point that satisfies these four conditions.

---

## Constrained Optimization — Framework

The key insight behind the Constrained Optimization Frameworks is simple: if violating a constraint is made sufficiently expensive, the optimizer will avoid violations on its own. Lectures 16–18 introduce three algorithms that compute a KKT point by transforming the constrained problem into a sequence of tractable subproblems.

| Method | Core Idea | Subproblem Type | Feasibility of Iterates | Convergence |
|--------|-----------|-----------------|------------------------|-------------|
| **Penalty Method** | Append a penalty for constraint violation | Unconstrained minimization | Iterates are infeasible | Exact KKT point recovered only in the limit; linear rate |
| **Barrier Method** | Append a barrier that blows up near constraint boundary | Unconstrained minimization | Iterates are strictly feasible | Exact KKT point recovered only in the limit; linear rate |
| **Interior-Point Method** | Solve KKT system directly with barrier modification; Newton steps inside feasible region | Equality-constrained Newton system | Iterates are strictly feasible | Exact KKT point recovered only in the limit; polynomial-time; superlinear rate |

## 2. Barrier Method

For the problem $\min f(x)$ subject to $g_i(x) \leq 0$, the Barrier Method introduces a **logarithmic barrier function** $B(x, \mu) = f(x) + \mu \sum_{i=1}^{m} \ln(-1/g_i(x))$. Here, the **barrier** $\ln(-1/g_i(x))$ is defined only when $g_i(x) < 0$ (strict feasibility) and tends to $+\infty$ as $g_i(x) \to 0^-$, creating an impenetrable wall at each constraint boundary, while the **barrier parameter** $\mu > 0$ controls the height of this wall.

In practice, the Barrier Method starts with a large barrier parameter $\mu^{(0)} > 0$ and iteratively solves a sequence of unconstrained subproblems $x^{(k)} = \arg\min B(x^{(k)}, \mu^{(k)})$ with a progressively smaller barrier parameter $\mu^{(k+1)} = \theta \cdot \mu^{(k)}$; $\theta \in (0, 1)$ until the barrier influence becomes negligible $\mu^{(k)} < \varepsilon$. Consequently, comparing the stationarity conditions of the unconstrained subproblem with that of the original constrained problem recovers the Lagrange multiplier for each active constraint in the limit $\lambda_i^* = \lim_{\mu \to 0} \ - \mu/g_i(x^*(\mu))$. Despite its polynomial-time convergence, the Barrier Method suffers from two practical limitations: (i) a strictly feasible starting point is required since the barrier function is undefined outside the feasible set, and (ii) as $\mu \to 0$, the Hessian of $B$ becomes increasingly ill-conditioned near the constraint boundary, requiring $\mu$ to be reduced gradually.

```{tip}
For the unconstrained subproblem: solve the system of equations — $\nabla B(x, \mu) = \mathbf{0}$ analytically if a closed-form solution can be established; otherwise, default to Quasi-Newton's Method; use Newton's Method when the Hessian is cheap to compute and invert; use Gradient Descent when problem scale makes storing a Hessian prohibitive.
```

```{note}
The Barrier Method approaches a KKT point from strictly inside the feasible region: its iterates always satisfy $g_i(x) < 0$ (strict inequality). As $\mu \to 0$, the barrier trajectory converges to the boundary of the active constraints, and the implicit multiplier $\mu / (-g_i(x^*(\mu)))$ converges to $\lambda_i^*$.
```

---

## In-Class Exercises

### Exercise 1

MTC Chennai is optimizing a demand-responsive feeder service on the Tambaram–Chromepet corridor. The daily operating cost depends on fleet size $n$ (vehicles) and scheduled headway $h$ (minutes):

$$C(n, h) = (n - 5)^2 + 2(h - 3)^2$$

The depot has a combined resource constraint: $n + h \leq 7$.

1. The unconstrained optimum is $(n^*, h^*) = (5, 3)$ with $n+h = 8 > 7$ — the constraint binds. Identify a strictly feasible starting point for the Barrier Method. Why does the Penalty Method not require this?
2. Formulate the **logarithmic barrier function** $B(n, h, \mu)$. Write out the stationarity conditions $\nabla B = \mathbf{0}$.
3. Let $s = 7 - n - h > 0$ denote the slack. Show that the stationarity conditions imply:
   - $n = 2h - 1$ (ratio condition, same as KKT)
   - $h^*(\mu) = \dfrac{17 - \sqrt{1 + 3\mu}}{6}$
4. Evaluate the barrier trajectory at $\mu = 1$ and $\mu = 0.1$. Fill in the table below, including the implicit multiplier $\hat{\lambda} = \mu / s$.
5. Verify the KKT conditions at the constrained optimum $\left(n^* = \tfrac{13}{3},\, h^* = \tfrac{8}{3}\right)$ and confirm that $\hat{\lambda} \to \lambda^* = \tfrac{4}{3}$ as $\mu \to 0$.

#### Barrier Trajectory Table

| $\mu$ | $n^*(\mu)$ | $h^*(\mu)$ | $n + h$ | Slack $s = 7 - n - h$ | Implicit $\hat{\lambda} = \mu / s$ |
|-------|-----------|-----------|---------|----------------------|-----------------------------------|
| $2$ | | | | | |
| $1$ | $4$ | $5/2$ | $13/2$ | $1/2$ | $2$ |
| $0.1$ | | | | | |
| $0.01$ | | | | | |
| $\mu \to 0$ (KKT) | $13/3$ | $8/3$ | $7$ | $0$ | $4/3$ |

*(The entry for $\mu = 1$ has been completed as a worked example.)*

### Exercise 2

An NHAI corridor study models total system travel time across two parallel links — the Mumbai–Pune Expressway ($v_1$, thousand veh/hr) and the old NH48 ($v_2$, thousand veh/hr):

$$T(v_1, v_2) = 3v_1^2 - 2v_1 v_2 + 2v_2^2 - 14v_1 - 12v_2 + 50$$

A combined capacity constraint limits total flow: $v_1 + v_2 \leq 5$.

1. Formulate the **logarithmic barrier function** $B(v_1, v_2, \mu)$ and write the stationarity conditions $\nabla B = \mathbf{0}$. Let $s = 5 - v_1 - v_2 > 0$ denote the slack.
2. Show that the stationarity conditions imply $v_2 = (4v_1 - 1)/3$ (the same ratio as in the KKT conditions from Lecture 16), and that the slack satisfies:

$$10s^2 + 40s - 7\mu = 0 \implies s^*(\mu) = \frac{-40 + \sqrt{1600 + 280\mu}}{20}$$

3. Compute $v_1^*(\mu)$ and $v_2^*(\mu)$ at $\mu = 1$ and $\mu = 0.1$. Does the barrier trajectory approach $(v_1^*, v_2^*) = (16/7,\, 19/7)$ as $\mu \to 0$?
4. Compute the implicit multiplier $\hat{\lambda} = \mu / s$ at each $\mu$ value and compare with the Penalty Method implicit multiplier from Lecture 16. What do you observe?
5. Compare the two methods: for the same target accuracy $|\hat{\lambda} - \lambda^*| < 0.05$, which method requires a smaller parameter value ($\rho$ or $\mu$)?

---

## Take-Away Exercises

### Exercise 1

Delhivery is minimizing the daily operating cost at its Chennai cross-dock facility. The cost depends on daily throughput $\lambda$ (tonnes/day) and staffing level $s$ (workers):

$$C(\lambda, s) = 2\lambda^2 - 4\lambda s + 3s^2 - 12\lambda - 6s + 50$$

A combined budget and shift constraint limits $\lambda + 2s \leq 14$.

1. Formulate the **logarithmic barrier function** $B(\lambda, s, \mu)$ and derive the stationarity conditions.
2. Let $t = 14 - \lambda - 2s > 0$ denote the slack. Show that the stationarity conditions imply $6\lambda - 7s = 9$ (same as KKT) and that $s^*(\mu)$ is the smaller root of:

$$19\,s^2 - 246\,s + (675 - 9\mu) = 0$$

3. Compute $\lambda^*(\mu)$ and $s^*(\mu)$ for the $\mu$ values in the table below. Fill in the slack $t$, the implicit multiplier $\hat{\lambda} = \mu / t$, and compare with the Penalty Method results from Lecture 16.
4. Verify that $\hat{\lambda} \to \lambda^* = \tfrac{64}{19}$ as $\mu \to 0$. Interpret this multiplier for the Delhivery operations manager.

#### Barrier Trajectory Table

| $\mu$ | $\lambda^*(\mu)$ | $s^*(\mu)$ | $\lambda + 2s$ | Slack $t$ | Implicit $\hat{\lambda} = \mu/t$ |
|-------|-----------------|-----------|---------------|-----------|----------------------------------|
| $4$ | | | | | |
| $2$ | | | | | |
| $1$ | | | | | |
| $0.1$ | | | | | |
| $0.01$ | | | | | |
| $\mu \to 0$ (KKT) | $116/19$ | $75/19$ | $14$ | $0$ | $64/19$ |

### Exercise 2

BMTC is optimizing its demand-responsive feeder service in Bengaluru. The total daily operating cost depends on fleet size $n$ (vehicles) and headway $h$ (minutes):

$$C(n, h) = 2n^2 + 3h^2 + 2nh - 20n - 30h + 100$$

A depot capacity constraint limits $n \leq 2$.

1. Formulate the **logarithmic barrier function** $B(n, h, \mu)$ for the constraint $n \leq 2$, written as $g(n) = n - 2 \leq 0$.
2. Derive the stationarity conditions $\nabla B = \mathbf{0}$. Show that $h^*(\mu) = (15 - n^*(\mu))/3$ (independent of $\mu$ once $n$ is fixed) and that $n^*(\mu)$ satisfies:

$$n^*(\mu) = \frac{5 - \sqrt{1 + 6\mu/5}}{2}$$

3. Implement the Barrier Method in Python using `scipy.optimize.minimize` with `method='BFGS'` to solve the barrier subproblem at each outer iteration. Use $\mu_0 = 2$, reduction factor $\theta = 0.1$, and stop when $\mu_k < 10^{-4}$. Start from $(n, h) = (1.5, 4.5)$ — verify this is strictly feasible.
4. Report $n^*(\mu)$, $h^*(\mu)$, the slack $2 - n^*(\mu)$, and the implicit multiplier $\hat{\lambda} = \mu / (2 - n^*(\mu))$ at each outer iteration. Compare with the Penalty Method results from Lecture 16, Exercise 2.
5. Interpret $\lambda^* = \tfrac{10}{3}$ for the BMTC operations manager. Is this interpretation the same as in Lecture 16? Should it be?

---

## Further Reading

- Nocedal, J. and Wright, S.J. (2006). *Numerical Optimization* (2nd ed.). Springer — Chapter 17 (barrier methods and the central path); Chapter 19 (interior-point methods for nonlinear programming).
- Bazaraa, M.S., Sherali, H.D., and Shetty, C.M. (2006). *Nonlinear Programming: Theory and Algorithms* (3rd ed.). Wiley — Chapter 12 (barrier and interior-point methods).
- Luenberger, D.G. and Ye, Y. (2016). *Linear and Nonlinear Programming* (4th ed.). Springer — Chapter 13 (penalty and barrier methods).
- SciPy documentation: `scipy.optimize.minimize` — [docs.scipy.org](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)